# A FAIR² Mental Health Survey Dataset Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display basic dataset info
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")
print(f"Identifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

We'll list the record sets, then enumerate the fields and columns in each using their `@id`s.

In [ ]:
# List available record sets by their `@id`

record_sets = metadata.recordSet
if not record_sets:
    print("No record sets found in metadata.")
else:
    for rs in record_sets:
        # Each record set is an object (usually mlcroissant.RecordSet)
        print(f"RecordSet ID: {rs['@id'] if isinstance(rs, dict) else rs._id}")
        print(f"  Name: {rs['name'] if isinstance(rs, dict) and 'name' in rs else getattr(rs, 'name', '(no name)')}")
        # Show fields
        fields = rs['field'] if isinstance(rs, dict) and 'field' in rs else getattr(rs, 'field', [])
        print("  Fields:")
        for f in fields:
            print(f"    Field ID: {f['@id'] if isinstance(f, dict) else f._id}")
            print(f"      Name: {f['name'] if isinstance(f, dict) and 'name' in f else getattr(f, 'name', '(no name)')}")
            # Show columns
            columns = f['column'] if isinstance(f, dict) and 'column' in f else getattr(f, 'column', [])
            for c in columns:
                print(f"      Column ID: {c['@id'] if isinstance(c, dict) else c._id}")
                print(f"        Name: {c['name'] if isinstance(c, dict) and 'name' in c else getattr(c, 'name', '(no name)')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We'll select available record sets and extract their records as DataFrames, referencing each by their `@id`.

In [ ]:
# Retrieve record set IDs
rs_ids = []
if metadata.recordSet:
    for rs in metadata.recordSet:
        rs_id = rs['@id'] if isinstance(rs, dict) else rs._id
        rs_ids.append(rs_id)
else:
    print("No record sets found.")

# Create DataFrames for each record set
dataframes = {}

for record_set_id in rs_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
    else:
        print(f"No records found for record set ID: {record_set_id}")

# Display columns for the first record set, if available
if dataframes:
    first_rs_id = rs_ids[0]
    print(f"RecordSet ID: {first_rs_id} Columns:")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()
else:
    print("No dataframes extracted.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records, normalizing numeric fields, and grouping by attributes. Operations include removing outliers, transforming distributions, or aggregating data, referencing fields/columns by their `@id`s.

In [ ]:
# Example: EDA on GAD-7 score field in primary record set
# Please update numeric_field_id and group_field_id with actual IDs from your dataset overview.

# Replace with correct record set and field IDs as available
primary_record_set_id = rs_ids[0] if rs_ids else None
df = dataframes.get(primary_record_set_id, None)

# Guessing at likely numeric and group fields based on typical survey naming
# Example guesses: 'gad7_score', 'phq9_score', 'age', 'gender', use exact @id from overview
numeric_field_id = None
group_field_id = None

# Try to find a numeric field
if df is not None:
    for col in df.columns:
        if 'score' in col.lower() or 'age' in col.lower():
            numeric_field_id = col
            break

    for col in df.columns:
        if 'gender' in col.lower() or 'education' in col.lower():
            group_field_id = col
            break

if df is not None and numeric_field_id is not None:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric fields found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of a numeric field and show group-wise boxplots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution for numeric field
if df is not None and numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Box plot for numeric_field by group_field
    if group_field_id is not None:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we:
- Loaded the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using its Croissant schema.
- Inspected record set and field identifiers via their `@id`, ensuring reproducible references.
- Extracted survey responses and performed basic exploratory analysis on numeric fields such as GAD-7 or PHQ-9 scores.
- Visualized distributions and relationships within the data.

These steps show how `mlcroissant` enables robust, standardized data access and processing. Further analysis can build upon these foundations for in-depth mental health and demographic research.